In [168]:
import time
import requests
import json
from pathlib import Path
import pandas as pd
from data import load_data

# ----- CONFIGURE THESE -----
procedure_data_path = r"D:\Python Projects\Hospital readmission risk\data\raw\dictionaries\unique_procedures.csv"
procedure_write_path = r"D:\Python Projects\Hospital readmission risk\data\processed\dictionaries\procedures_dictionary.csv"
diagnoses_data_path = r"D:\Python Projects\Hospital readmission risk\data\raw\dictionaries\unique_diagnoses.csv"
diagnoses_write_path = r"D:\Python Projects\Hospital readmission risk\data\processed\dictionaries\diagnoses_dictionary.csv"
concept_path = r"D:\Python Projects\Hospital readmission risk\data\concepts"
procedure_sql = """
    select * 
    from `raw_data_for_dictionaries.unique_procedures`
"""
diagnoses_sql = """
    select *
    from `raw_data_for_dictionaries.unique_diagnoses`
"""
BASE = "https://termbrowser.nhs.uk/sct-browser-api/snomed"
EDITION = "uk-edition"
RELEASE = "v20251119"  # keep in sync with what you see in the browser
MAX_RETRIES = 3
BACKOFF_SECONDS = 2.0
REQUEST_COUNT = 0
STATE_PATH = Path("D:\Python Projects\Hospital readmission risk\data\intermediate\diagnosess_snomed_state.json")
#STATE_PATH = Path("D:\Python Projects\Hospital readmission risk\data\intermediate\procedures_snomed_state.json")
CACHE: dict[str, str] = {}
RESULTS: dict[str, dict[str, bool]] = {} 
procedure_targets = {    
    "is_procedure": {
        "243796009", #situation
        "14734007", #admin_procedure
        "409073007", #education_procedure
        "108217004", #interview, history, physical examination
        "119270007", #management procedure
        "185316007", #indirect encounter
        "122869004", #measurement procedure
        "363787002", #observation
        "404684003", #clinical finding
        "72353004", #psychosocial
        "260787004", #physical_object
        "11429006" #consultation
                    },
    "is_therapy" : {"243120004"},
    "is_surgery" : {"387713003"},
    "is_planning" : {"410538000", "183976008"}
}    
diagnosis_targets = {
    "is_disorder" : {"64572001"},
    "is_symptom" : {"404684003"},
    "is_dementia" : {"52448006"},
    "is_cancer" : {"363346000"},
    "is_hiv": {"86406008"},
    "is_hf": {"105981003"},
    "is_ckd": {"709044004"},
    "is_diabetes": {"44054006"},
    "is_lf" : {"235856003"},
    "is_chronic": {"27624003"},
    'inflammation': {'363171009'},
    'musculoskeletal': {'928000'},
    'nervous': {'118940003'},
    'respiratory': {'50043002'},
    'cardiac': {'49601007'},
    'renal': {'90708001'},
    'trauma': {'417163006'},
    'intoxication': {'1149322001'}
}

    # concept_id -> True/False results
session = requests.Session()
# ----------------------------

In [169]:
def load_state():

    global CACHE, RESULTS

    if not STATE_PATH.exists():
        return

    with STATE_PATH.open("r", encoding="utf-8") as f:
        data = json.load(f)

    raw_cache = data.get("cache", {})
    CACHE = {code: path for code, path in raw_cache.items()}
    """
    raw_results = data.get("results", {})  # dict[str, dict]
    RESULTS = {
        code: {flag: bool(val) for flag, val in flags.items()}
        for code, flags in raw_results.items()
            }
            """
            

In [170]:
def save_state():

    data = {
        "cache": CACHE,
        "results": RESULTS,
    }
    
    with STATE_PATH.open("w", encoding="utf-8") as f:
        json.dump(data, f)

In [171]:
def save_concept(concept_id, data, save_path = concept_path):

    global REQUEST_COUNT

    path = Path(save_path)
    path.mkdir(parents=True, exist_ok=True)

    file_name = f"{concept_id}.json"
    file_path = path/file_name

    with file_path.open("w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

    CACHE[concept_id] = str(file_path)

    REQUEST_COUNT = REQUEST_COUNT + 1

    if REQUEST_COUNT % 100 == 0:
        save_state()


In [172]:
def read_concept(concept_path):

    path = Path(concept_path)

    if not path.exists():
        return {}
    
    with path.open("r", encoding="utf-8") as f:
        return json.load(f) 

In [173]:

def get_concept(concept_id: str, targets) -> dict:
    """Fetch a SNOMED concept JSON once, with simple 429 handling and caching."""

    if concept_id in CACHE:

        return read_concept(CACHE[concept_id])

    url = f"{BASE}/{EDITION}/{RELEASE}/concepts/{concept_id}"

    for attempt in range(1, MAX_RETRIES + 1):

        resp = session.get(url)

        if resp.status_code == 429:
            # too many requests - small backoff then retry
            time.sleep(BACKOFF_SECONDS * attempt)
            continue

        if resp.status_code in (404, 410, 500, 502, 503, 504):

            save_concept(concept_id, {})

            for flag in targets:
                RESULTS[flag][concept_id] = False
                
            return {}

        resp.raise_for_status()

        data = resp.json()

        save_concept(concept_id, data)
        
        return data

    # if we get here, all retries hit 429 or other non-OK without raise
    resp.raise_for_status()

In [174]:
def get_parent_ids(concept: dict) -> list[str]:

    """Extract parent conceptIds from the concept JSON (adjust to actual structure)."""

    parents = []

    for rel in concept.get("relationships", []):
        # 'is a' relationship has conceptId 116680003 in SNOMED CT
        if rel.get("type", {}).get("conceptId") == "116680003" and rel.get("active"):

            target = rel.get("target") or {}

            cid = target.get("conceptId")

            if cid:
                
                parents.append(cid)
            
    return parents

In [175]:
def is_or_has_ancestor_in(concept_id: str, targets, target_ids: set[str], flag: str, max_depth: int = 10) -> bool:
    # if we’ve already fully explored this concept’s ancestors in a previous run,
    # just return the previous answer if you also store it, or skip re-walk.

    if flag not in RESULTS: 
        RESULTS[flag] = {}

    if concept_id in RESULTS[flag]:
        return RESULTS[flag][concept_id]

    visited = set()
    frontier = {concept_id}
    result = False

    if concept_id in target_ids:
        RESULTS[flag][concept_id] = True
        return True

    for depth in range(max_depth):

        next_frontier = set()

        for cid in frontier:

            if cid in visited:
                continue

            if cid in RESULTS[flag] and RESULTS[flag][cid]:
                #save_state()
                RESULTS[flag][concept_id] = True
                return True

            visited.add(cid)

            c = get_concept(cid, targets)

            if not isinstance(c, dict):
                continue

            parents = get_parent_ids(c)

            if target_ids.intersection(parents):

                RESULTS[flag][concept_id] = True
                
                if cid != concept_id:
                    RESULTS[flag][cid] = True

                return True
            
            next_frontier.update(p for p in parents if p not in visited)

        if not next_frontier:
            break
        
        frontier = next_frontier

    RESULTS[flag][concept_id] = False

    return result

In [176]:
def build_dictionary(data, targets):

    for id in data['code']:

        for flag in targets:

            b = is_or_has_ancestor_in(str(id), targets, targets[flag], flag)
        

In [177]:
def build_flags(data: pd.DataFrame):

    data.set_index('code', inplace = True)

    for flag in RESULTS:
        
        col = pd.Series(RESULTS[flag], name = flag)
        col.index = col.index.astype(int)
        data[flag] = col

        if flag == 'is_procedure':
                data[flag] = ~data[flag]

    cols = [col for col in data.columns if not col.startswith("name")]
    data[cols] = data[cols].map(lambda x: int(x))


In [178]:
def get_description(concept_id: str):

    if CACHE[concept_id]:

        return get_concept(concept_id, diagnosis_targets)["descriptions"][0]["term"]

    return None


In [179]:
def fill_descriptions(data):

    description_na_mask = data['name'].isna()

    for id in data.loc[description_na_mask, :].index:
         
        data.loc[id, 'name'] = get_description(str(id))

In [180]:
def fix_disorders(data):

    disorder_mask = data['is_disorder'] == 1
    data.loc[disorder_mask, 'is_symptom'] = 0

In [181]:
def fix_chronic(data):

    chronic_mask = data.loc[:, 'is_dementia':'is_lf'].sum(axis = 1) > 0
    data.loc[chronic_mask, 'is_chronic'] = 1

In [182]:
def fix_flags(data):

    fix_disorders(data)
    fix_chronic(data)

In [183]:
def pack_dictionary(data, path: str):

    data.to_csv(path)

In [184]:
if __name__ == "__main__":
    """
    Running checklist:
    1. Comment wrong path in STATE_PATH at the beginning
    2. In load_state() comment results if rebuilding flags
    3. Here comment all that is relevant to another dictionary
    """


    load_state()
    
    #data = load_data(procedure_data_path, procedure_sql)           # load previous results

    data = load_data(diagnoses_data_path, diagnoses_sql)
    
    try:
        
        #build_dictionary(data, procedure_targets)
        build_dictionary(data, diagnosis_targets)

    finally:
        save_state()

        
    build_flags(data)

    fill_descriptions(data)

    #pack_dictionary(data, procedure_write_path) 
    """
    Always Sanity check before packing a dictionary
    """
    fix_flags(data)
    #pack_dictionary(data, diagnoses_write_path)

In [185]:
data

,name,is_disorder,is_symptom,is_dementia,is_cancer,is_hiv,is_hf,is_ckd,is_diabetes,is_lf,is_chronic,inflammation,musculoskeletal,nervous,respiratory,cardiac,renal,trauma,intoxication
code,,,,,,,,,,,,,,,,,,,
1734006,Fracture of vertebral column with spinal cord ...,1,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,1,0
3718001,Cow's milk,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4557003,Preinfarction syndrome,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0
5602001,Opioid abuse,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
6072007,Bleeding from anus,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
157141000119108,Proteinuria due to type 2 diabetes mellitus,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0
368581000119106,Neuropathy due to type 2 diabetes mellitus,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0
442571000124108,Tree nut (substance),0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [186]:
pack_dictionary(data, diagnoses_write_path)

---

In [ ]:
build_flags(data)
fill_descriptions(data, 'diagnosis_name')
pack_dictionary(data)

KeyError: 'description'

In [126]:
load_state()
data = load_data(diagnoses_data_path, diagnoses_sql)

In [139]:
data

,name,trauma
code,,
1734006,Fracture of vertebral column with spinal cord ...,1
3718001,NaN,0
4557003,Preinfarction syndrome,1
5602001,NaN,0
6072007,Bleeding from anus,0
...,...,...
157141000119108,Proteinuria due to type 2 diabetes mellitus,0
368581000119106,Neuropathy due to type 2 diabetes mellitus,0
442571000124108,NaN,0


In [166]:
data = load_data(diagnoses_data_path, diagnoses_sql)
targets = {'pregnancy': {'248982007'}} 
#traumatic or non-traaumatic injury and wound finding

build_dictionary(data, targets)
build_flags(data)

In [153]:
main_diagnoses = pd.read_csv(r"D:\Python Projects\Hospital readmission risk\data\processed\main_diagnoses.csv")

In [154]:
main_diagnoses

,stay_id,main_diagnosis_code,main_diagnosis_name,main_diagnosis_type,num_of_disorders,num_of_findings
0,bb6871e0-62bb-8c5c-db55-b301aa1bab7a,NaN,NaN,NaN,0,0
1,bb7e1e13-de9e-c0ff-95bd-6836f4469324,NaN,NaN,NaN,0,0
2,bb83ff80-9b9d-d277-941d-47e98370a9be,NaN,NaN,NaN,0,0
3,bcf479a6-a5ab-b7d0-01f0-e32a512716cc,NaN,NaN,NaN,0,0
4,bdcfa2bb-1058-7631-9197-d843fae92eca,NaN,NaN,NaN,0,0
...,...,...,...,...,...,...
67913,308d7cfc-1924-b77c-4706-ebe6d7da8379,3.685810e+14,Neuropathy due to type 2 diabetes mellitus (di...,disorder,1,0
67914,0e671f61-cb82-7e99-6eb0-fac048867c21,6.638301e+07,Gingivitis,disorder,1,0
67915,182b1087-271f-112f-0611-281077567630,6.638301e+07,Gingivitis,disorder,1,0
67916,2a21162c-c320-0d74-0ae2-7710c1efcf79,NaN,NaN,NaN,0,0


In [155]:
main_diagnoses.set_index('stay_id', inplace = True)

In [167]:
for row in main_diagnoses.index:

    if not pd.isna(main_diagnoses.loc[row, 'main_diagnosis_code']):

        code = main_diagnoses.loc[row, 'main_diagnosis_code']
        main_diagnoses.loc[row, 'pregnancy'] = int(data.loc[code, 'pregnancy'])

main_diagnoses['pregnancy'].sum()

np.float64(4609.0)

In [157]:
disorders = main_diagnoses[main_diagnoses['main_diagnosis_type'] == 'disorder']
findings = main_diagnoses[main_diagnoses['main_diagnosis_type'] == 'finding']

In [146]:
print(f"Disorders: {disorders['trauma'].sum()}, findings: {findings['trauma'].sum()}")

Disorders: 11304.0, findings: 0.0


In [158]:
disorders

,main_diagnosis_code,main_diagnosis_name,main_diagnosis_type,num_of_disorders,num_of_findings,trauma
stay_id,,,,,,
bc547828-5aab-faa1-0fa3-3977e9a3d769,4.013030e+08,Acute ST segment elevation myocardial infarcti...,disorder,2,0,1.0
bfc30598-f78e-2c23-cf98-d6c9ae85dbbf,4.013030e+08,Acute ST segment elevation myocardial infarcti...,disorder,2,0,1.0
c370dacb-3b10-9e31-bce8-0cc74868f011,4.013140e+08,Acute non-ST segment elevation myocardial infa...,disorder,2,0,1.0
c4f826b2-fefe-336f-7105-9c60815b5ba2,4.557003e+06,Preinfarction syndrome,disorder,2,0,1.0
c5f56bd5-5cb8-db3f-11c2-2e5d7090d057,4.013140e+08,Acute non-ST segment elevation myocardial infa...,disorder,2,0,1.0
...,...,...,...,...,...,...
3c18f46b-cc1b-8a6d-c874-d0e09a808bdb,1.241710e+14,Chronic intractable migraine without aura (dis...,disorder,1,0,0.0
2d56633f-71fd-d363-ca5f-3620a6909435,4.617700e+07,End stage renal disease,disorder,3,1,0.0
308d7cfc-1924-b77c-4706-ebe6d7da8379,3.685810e+14,Neuropathy due to type 2 diabetes mellitus (di...,disorder,1,0,0.0


In [159]:
diagnoses_dictionary = pd.read_csv(diagnoses_write_path)

In [160]:
diagnoses_dictionary

,code,name,is_disorder,is_symptom,is_dementia,is_cancer,is_hiv,is_hf,is_ckd,is_diabetes,is_lf,is_chronic,inflammation,musculoskeletal,mental,nervous,respiratory,cardiac,renal
0,1734006,Fracture of vertebral column with spinal cord ...,1,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0
1,3718001,Cow's milk,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,4557003,Preinfarction syndrome,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0
3,5602001,Opioid abuse,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0
4,6072007,Bleeding from anus,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
331,157141000119108,Proteinuria due to type 2 diabetes mellitus,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1
332,368581000119106,Neuropathy due to type 2 diabetes mellitus,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0
333,442571000124108,Tree nut (substance),0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
334,453131000124105,Videotelephony encounter (procedure),0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [163]:
disorders = diagnoses_dictionary[diagnoses_dictionary['is_disorder'] == 1]
disorders['marked'] = disorders.loc[:, 'inflammation':'renal'].sum(axis = 1)
disorders[disorders['marked'] == 0]

C:\Users\4infi\AppData\Local\Temp\ipykernel_10156\2370332310.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  disorders['marked'] = disorders.loc[:, 'inflammation':'renal'].sum(axis = 1)


,code,name,is_disorder,is_symptom,is_dementia,is_cancer,is_hiv,is_hf,is_ckd,is_diabetes,is_lf,is_chronic,inflammation,musculoskeletal,mental,nervous,respiratory,cardiac,renal,marked
4,6072007,Bleeding from anus,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
8,11218009,Infection caused by Pseudomonas aeruginosa,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
14,18718003,Gingival disease,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
15,19169002,Miscarriage in first trimester,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
21,27942005,Shock,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
311,1163220007,Pressure injury stage II,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
317,1501000119109,Proliferative diabetic retinopathy associated ...,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
318,1551000119108,Nonproliferative diabetic retinopathy associat...,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
320,60951000119105,Blindness due to type 2 diabetes mellitus,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
